##Quantum vs Classical ML Classification Comparison

In [1]:
# Install required packages for Google Colab
!pip install qiskit qiskit-machine-learning seaborn qiskit-ibm-runtime matplotlib scikit-learn qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.9/363.9 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 13.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing

In [2]:
# Installing cuQuantum (NVIDIA GPU acceleration)
# requires CUDA-compatible GPU
try:
    !pip install cuquantum-python
    !pip install qiskit-aer-gpu
    CUQUANTUM_AVAILABLE = True
    print("cuQuantum installation successful")
except:
    print("cuQuantum not available - using CPU simulation")
    CUQUANTUM_AVAILABLE = False

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 MB 5.4 MB/s eta 0:00:00
  Created wheel for cuquantum-python: filename=cuquantum_python-25.3.0-0_cu12-py3-none-any.whl size=6621 sha256=381bee417ca36ef9a9bee1a14346bac971406fa5aebccad64cc6e7d1c148a95f
  Stored in directory: /root/.cache/pip/wheels/c1/66/53/4f7b83db2366a1eca8ce7058ee072183c84a32d3978c1a9b65
Successfully built cuquantum-python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 103.2 MB/s eta 0:00:00
cuQuantum installation successful


Import Libraries

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, n_local
from qiskit_machine_learning.optimizers import SPSA, COBYLA
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.datasets import ad_hoc_data
from qiskit.primitives import StatevectorSampler
from qiskit_aer import AerSimulator

print("Basic imports completed")

Basic imports completed


In [4]:
# IBM Quantum imports
try:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    IBM_AVAILABLE = True
    print("IBM Runtime available")
except ImportError:
    print("IBM Runtime not available")
    IBM_AVAILABLE = False

# Classical ML imports
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

print("Classical ML imports completed")

IBM Runtime available
Classical ML imports completed


In [5]:
# cuQuantum imports
if CUQUANTUM_AVAILABLE:
    try:
        from qiskit_aer.backends import AerSimulator
        import cupy as cp
        print("cuQuantum ready for acceleration")
    except ImportError:
        print("cuQuantum imports failed, falling back to CPU")
        CUQUANTUM_AVAILABLE = False
else:
    print("cuQuantum not available")

cuQuantum ready for acceleration


Dataset preparation

In [6]:
# Set random seed for reproducibility
seed = 1376
algorithm_globals.random_seed = seed
np.random.seed(seed)

# Dataset parameters
feature_dim = 2  # Dimension of each data point
training_size = 40  # Increased for better training
test_size = 20

print("Generating dataset...")
# Generate ad hoc dataset for classification
training_features, training_labels, test_features, test_labels = ad_hoc_data(
    training_size=training_size,
    test_size=test_size,
    n=feature_dim,
    gap=0.3
)

# Convert labels from one-hot to single labels for classical ML
training_labels_classical = np.argmax(training_labels, axis=1)
test_labels_classical = np.argmax(test_labels, axis=1)

print(f"Training samples: {len(training_features)}")
print(f"Test samples: {len(test_features)}")
print(f"Feature dimension: {feature_dim}")

Generating dataset...
Training samples: 80
Test samples: 40
Feature dimension: 2


Classical Machine Learning Baselines

In [7]:
# Standardize features for classical ML
scaler = StandardScaler()
training_features_scaled = scaler.fit_transform(training_features)
test_features_scaled = scaler.transform(test_features)

# Define classical ML models
classical_models = {
    'Logistic Regression': LogisticRegression(random_state=seed, max_iter=1000),
    'SVM (RBF)': SVC(kernel='rbf', random_state=seed),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=seed)
}

classical_results = {}

for name, model in classical_models.items():
    print(f"\nTraining {name}...")

    # Measure training time
    start_time = time.time()
    model.fit(training_features_scaled, training_labels_classical)
    training_time = time.time() - start_time

    # Measure prediction time
    start_time = time.time()
    predictions = model.predict(test_features_scaled)
    prediction_time = time.time() - start_time

    # Calculate accuracy
    accuracy = accuracy_score(test_labels_classical, predictions)

    classical_results[name] = {
        'accuracy': accuracy,
        'training_time': training_time,
        'prediction_time': prediction_time,
        'total_time': training_time + prediction_time
    }

    print(f"Accuracy: {accuracy:.3f}")
    print(f"Training time: {training_time:.3f}s")
    print(f"Prediction time: {prediction_time:.4f}s")

print("\nClassical ML baseline completed!")


Training Logistic Regression...
Accuracy: 0.525
Training time: 0.019s
Prediction time: 0.0004s

Training SVM (RBF)...
Accuracy: 0.625
Training time: 0.008s
Prediction time: 0.0007s

Training Random Forest...
Accuracy: 0.625
Training time: 0.258s
Prediction time: 0.0203s

Classical ML baseline completed!


Quantum Circuit Setup

In [9]:
# Creating feature map and ansatz for VQC
feature_map = zz_feature_map(feature_dimension=feature_dim, reps=2, entanglement="linear")
ansatz = n_local(feature_map.num_qubits, ["ry", "rz"], "cz", reps=3)

qc_combined = feature_map.compose(ansatz)

Classical Quantum Simulator

In [10]:
# Use noise-resistant SPSA optimizer instead of COBYLA
optimizer = SPSA(maxiter=30)  # Reduced iterations for faster execution

# Classical simulator using StatevectorSampler
print("Setting up classical quantum simulator...")
classical_sampler = StatevectorSampler()

start_time = time.time()
vqc_classical = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    sampler=classical_sampler,
)

print("Training VQC on classical simulator...")
vqc_classical.fit(training_features, training_labels)
classical_training_time = time.time() - start_time

# Test the classical simulator
start_time = time.time()
classical_score = vqc_classical.score(test_features, test_labels)
classical_prediction_time = time.time() - start_time

print(f"\nClassical Simulator Results:")
print(f"Testing accuracy: {classical_score:.3f}")
print(f"Training time: {classical_training_time:.3f}s")
print(f"Prediction time: {classical_prediction_time:.3f}s")

Setting up classical quantum simulator...
Training VQC on classical simulator...

Classical Simulator Results:
Testing accuracy: 1.000
Training time: 62.446s
Prediction time: 0.288s


cuQuantum GPU Accelerated Simulator

In [11]:
cuquantum_results = {}

if CUQUANTUM_AVAILABLE:
    try:
        # Setup cuQuantum accelerated simulator
        print("Setting up cuQuantum GPU simulator...")

        # Use AerSimulator with GPU acceleration
        gpu_simulator = AerSimulator(method='statevector', device='GPU')

        # Create sampler for GPU simulation
        from qiskit.primitives import BackendSampler
        gpu_sampler = BackendSampler(gpu_simulator)

        # Use the same optimizer for fair comparison
        optimizer_gpu = SPSA(maxiter=30)

        start_time = time.time()
        vqc_gpu = VQC(
            feature_map=feature_map,
            ansatz=ansatz,
            optimizer=optimizer_gpu,
            sampler=gpu_sampler,
            callback=print
        )

        print("Training VQC on cuQuantum GPU simulator...")
        vqc_gpu.fit(training_features, training_labels)
        gpu_training_time = time.time() - start_time

        # Test the GPU simulator
        start_time = time.time()
        gpu_score = vqc_gpu.score(test_features, test_labels)
        gpu_prediction_time = time.time() - start_time

        cuquantum_results = {
            'accuracy': gpu_score,
            'training_time': gpu_training_time,
            'prediction_time': gpu_prediction_time,
            'total_time': gpu_training_time + gpu_prediction_time
        }

        print(f"\ncuQuantum GPU Results:")
        print(f"Testing accuracy: {gpu_score:.3f}")
        print(f"Training time: {gpu_training_time:.3f}s")
        print(f"Prediction time: {gpu_prediction_time:.3f}s")
        print(f"Speedup vs Classical: {classical_training_time/gpu_training_time:.2f}x")

    except Exception as e:
        print(f"cuQuantum GPU simulation failed: {e}")
        print("Falling back to CPU simulation")
        CUQUANTUM_AVAILABLE = False

Setting up cuQuantum GPU simulator...
Training VQC on cuQuantum GPU simulator...

cuQuantum GPU Results:
Testing accuracy: 0.900
Training time: 36.949s
Prediction time: 0.191s
Speedup vs Classical: 1.69x
